# Unsupervised Anomaly Detection — Isolation Forest

Building an unsupervised model to detect PortScan activity without using labels during 
training. Trained purely on Monday's clean benign baseline, then tested on Friday's 
PortScan data to see if the model can correctly flag attack traffic as anomalous.

**Approach:** train on benign-only data (Monday), predict on mixed data (Friday), then 
compare predictions against the real Label column — only used for evaluation, never for 
training.

#### Full plan
1. Setup & Load Data
2. Load Monday's Cleaned Data (training data)
3. Load Friday's Cleaned Data (testing data)
4. Align Columns Between Both Files
5. Prepare Features (drop Label, separate X and y)
6. Train Isolation Forest on Monday (benign only)
7. Predict on Friday's Data
8. Bring Back Real Labels & Evaluate
9. Visualize Results
10. Summary of Findings

## Step 1: Setup & Load Data

In [2]:
## Setup

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

## Step 2: Loading Monday's cleaned data( Training dataset)

In [3]:
## Load Monday's Cleaned Data (training data)

df_monday = pd.read_csv('/Users/chima/SecurITe_project/monday_clean.csv')
df_monday.shape

(528831, 79)

## Step 3:  Load Friday's Cleaned Data (testing data)

In [5]:
df_friday = pd.read_csv('/Users/chima/SecurITe_project/friday_portscan_clean.csv')
df_friday.shape
df_friday['Label'].value_counts()

Label
PortScan    158804
BENIGN      127041
Name: count, dtype: int64

## Step 4:Check if columns actually match

In [6]:
## Aligning Columns Between Both Files

set(df_monday.columns) == set(df_friday.columns)

True

## Step 5: Prepare Features — separate X (features) from y (labels)

In [7]:
 ## Preparing Features

# Monday: training data - dropping the label colm
X_train = df_monday.drop(columns=['Label'])

# Friday: testing data - separate features from the real labels
X_test = df_friday.drop(columns=['Label'])
y_test = df_friday['Label']

X_train.shape, X_test.shape, y_test.shape

((528831, 78), (285845, 78), (285845,))

X_train is Monday's data with the Label column removed.
X_test is Friday's data, also with Label removed this is what we gonna feed teh trained model to make predictions.
y_test is Friday's real Label column, kept completely separate by the end will use it toc heck the match.

## Checking for the column order : it must match exactly

In [8]:
# make sure X_train and X_test have columns in the exact same order
X_test = X_test[X_train.columns]
X_train.shape, X_test.shape

((528831, 78), (285845, 78))

## Step 6: Train Isolation Forest on Monday's data

In [9]:
## Training Isolation Forest on Monday (benign only)

model = IsolationForest(
    n_estimators=100,
    contamination='auto',
    random_state=42
)

model.fit(X_train)

,n_estimators,100
,max_samples,'auto'
,contamination,'auto'
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


## Step 7: Predict on Friday's data

In [11]:
predictions = model.predict(X_test)
predictions[:50]

array([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1, -1, -1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,  1, -1])

Isolation Forest doesn't return "BENIGN" or "PortScan" as itd enot know the labels
1 means "this looks normal" (similar to what it learned from Monday)
-1 means "this looks like an anomaly" (different from what it learned)

### how many rows did it flag as anomalies overall?

In [12]:
pd.Series(predictions).value_counts()

 1    274607
-1     11238
Name: count, dtype: int64

Out of 285,845 Friday rows, the model flagged only 11,238 as anomalies (-1), and called 274,607 normal (1).

But from my EDA on friday i know that there are actually 158804 real port scan rows but here the flagged rows are less huge gap.

In [ ]:
## Step 8: Bringing back the real labels and evaluating

In [24]:
## Evaluating Against Real Labels

results = pd.DataFrame({
    'actual_label': y_test.values,
    'prediction': predictions
})

# convert model's 1/-1 into something readable, side by side with real labels
results['predicted_label'] = results['prediction'].map({1: 'Normal', -1: 'Anomaly'})
results.head(30)

,actual_label,prediction,predicted_label
0,BENIGN,1,Normal
1,BENIGN,1,Normal
2,BENIGN,1,Normal
3,BENIGN,1,Normal
4,BENIGN,1,Normal
5,BENIGN,1,Normal
6,BENIGN,1,Normal
7,BENIGN,1,Normal
8,BENIGN,1,Normal
9,BENIGN,1,Normal


### cross-check — did the model's "Anomaly" flags line up with real PortScan rows?

In [18]:
## Confusion Matrix 

confusion_summary = pd.DataFrame({
    'Actual Label': ['BENIGN', 'BENIGN', 'PortScan', 'PortScan'],
    'Predicted As': ['Normal', 'Anomaly', 'Normal', 'Anomaly'],
    'Count': [
        ((results['actual_label']=='BENIGN') & (results['predicted_label']=='Normal')).sum(),
        ((results['actual_label']=='BENIGN') & (results['predicted_label']=='Anomaly')).sum(),
        ((results['actual_label']=='PortScan') & (results['predicted_label']=='Normal')).sum(),
        ((results['actual_label']=='PortScan') & (results['predicted_label']=='Anomaly')).sum(),
    ]
})
confusion_summary

,Actual Label,Predicted As,Count
0,BENIGN,Normal,116004
1,BENIGN,Anomaly,11037
2,PortScan,Normal,158603
3,PortScan,Anomaly,201


116,004 real BENIGN rows were correctly called "Normal" : good
11,037 real BENIGN rows were wrongly called "Anomaly" — false alarms
158,603 real PortScan rows were wrongly called "Normal" — the model completely missed these real attacks
only 201 real PortScan rows were correctly caught as "Anomaly" : good, but tiny

## Calculating precision and recall 

In [16]:
from sklearn.metrics import classification_report

# convert real labels into the same 1/-1 style the model uses, so we can compare directly
# treat PortScan as the "anomaly" we're trying to catch
y_test_binary = y_test.map({'BENIGN': 1, 'PortScan': -1})

print(classification_report(y_test_binary, predictions, target_names=['Anomaly (PortScan)', 'Normal (BENIGN)']))

                    precision    recall  f1-score   support

Anomaly (PortScan)       0.02      0.00      0.00    158804
   Normal (BENIGN)       0.42      0.91      0.58    127041

          accuracy                           0.41    285845
         macro avg       0.22      0.46      0.29    285845
      weighted avg       0.20      0.41      0.26    285845



PortScan detection — precision 0.02, recall 0.00. This means: 
of everything the model called "anomaly," only 2% actually turned out to be a real PortScan. 
And of all the real PortScan attacks that existed, the model caught essentially 0% of them.

BENIGN detection — precision 0.42, recall 0.91.
The model was decent at recognizing normal traffic as normal (91% recall), but not great at precision.
means the real portscan rows got mislabled as normal tooo

Overall accuracy: 0.41 (41%)

## Re-train with a more realistic contamination value
## Attempt 2 — Adjusting Contamination

The default contamination='auto' resulted in very poor recall (0.00) for PortScan detection, 
since it assumed anomalies were rare — but my EDA showed PortScan makes up roughly 55% of 
Friday's traffic. Trying contamination=0.3 as a more realistic starting point.

In [19]:
## Re-training with adjusted contamination

model_v2 = IsolationForest(
    n_estimators=100,
    contamination=0.3,
    random_state=42
)

model_v2.fit(X_train)

,n_estimators,100
,max_samples,'auto'
,contamination,0.3
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [20]:
## Predicting with adjusted model

predictions_v2 = model_v2.predict(X_test)
pd.Series(predictions_v2).value_counts()

 1    249088
-1     36757
Name: count, dtype: int64

In [21]:
## Confusion Matrix - Adjusted Model

results_v2 = pd.DataFrame({
    'actual_label': y_test.values,
    'prediction': predictions_v2
})
results_v2['predicted_label'] = results_v2['prediction'].map({1: 'Normal', -1: 'Anomaly'})

confusion_summary_v2 = pd.DataFrame({
    'Actual Label': ['BENIGN', 'BENIGN', 'PortScan', 'PortScan'],
    'Predicted As': ['Normal', 'Anomaly', 'Normal', 'Anomaly'],
    'Count': [
        ((results_v2['actual_label']=='BENIGN') & (results_v2['predicted_label']=='Normal')).sum(),
        ((results_v2['actual_label']=='BENIGN') & (results_v2['predicted_label']=='Anomaly')).sum(),
        ((results_v2['actual_label']=='PortScan') & (results_v2['predicted_label']=='Normal')).sum(),
        ((results_v2['actual_label']=='PortScan') & (results_v2['predicted_label']=='Anomaly')).sum(),
    ]
})
confusion_summary_v2

,Actual Label,Predicted As,Count
0,BENIGN,Normal,92362
1,BENIGN,Anomaly,34679
2,PortScan,Normal,156726
3,PortScan,Anomaly,2078


In [22]:
## Comparing v1 vs v2

comparison = pd.DataFrame({
    'Metric': ['PortScan caught (Anomaly)', 'PortScan missed (Normal)', 
               'BENIGN false alarms (Anomaly)', 'BENIGN correct (Normal)'],
    'v1 (auto)': [201, 158603, 11037, 116004],
    'v2 (0.3)': [2078, 156726, 34679, 92362]
})
comparison

,Metric,v1 (auto),v2 (0.3)
0,PortScan caught (Anomaly),201,2078
1,PortScan missed (Normal),158603,156726
2,BENIGN false alarms (Anomaly),11037,34679
3,BENIGN correct (Normal),116004,92362


In [23]:
## Recall Comparison (%)

recall_v1 = 201 / 158804 * 100
recall_v2 = 2078 / 158804 * 100

print(f"v1 recall: {recall_v1:.2f}%")
print(f"v2 recall: {recall_v2:.2f}%")

v1 recall: 0.13%
v2 recall: 1.31%


#### conclusion

After i made the model bit more senitive it caught 10 times more real PortScan attacks than before.
But it also created way more false alarms on normal traffic.

This model is not perfoming well i think Forest works best when anomalies are rare and scattered.
But when I visualized PortScan earlier in my EDA, it didn't look scattered at all it showed up as a big, tight, packed-together group of its own.

Maybe the mdoel is strggling bcz Portscan bhevaes like normal and not like outlier :may be wrong model for this setup.

## DBSCAN

#### So my plan is to try a different kind of model next — something called DBSCAN, which works by finding groups or clusters in the data, rather than looking for lonely outliers.

## Step 1: Prepare a manageable sample from Friday's data

In [25]:
## DBSCAN - Preparing Data

# DBSCAN can be slow on very large datasets, so we'll use a balanced sample first
sample_benign = df_friday[df_friday['Label'] == 'BENIGN'].sample(10000, random_state=42)
sample_portscan = df_friday[df_friday['Label'] == 'PortScan'].sample(10000, random_state=42)
df_dbscan_sample = pd.concat([sample_benign, sample_portscan])

X_dbscan = df_dbscan_sample.drop(columns=['Label'])
y_dbscan = df_dbscan_sample['Label']

X_dbscan.shape

(20000, 78)

## Step 2: Scale the features :DBSCAN DOES need scalig before 

In [29]:
## Scaling Features 

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_dbscan_scaled = scaler.fit_transform(X_dbscan)

## Step 3: Run DBSCAN

In [30]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=3, min_samples=10)
clusters = dbscan.fit_predict(X_dbscan_scaled)

pd.Series(clusters).value_counts()

 1     15799
 4      1153
-1      1094
 2      1005
 5       240
 3       122
 0       111
 7       111
 9        80
 22       26
 23       26
 14       25
 10       24
 11       21
 15       20
 12       17
 16       17
 20       17
 18       16
 13       14
 21       14
 8        13
 19       13
 17       12
 6        10
Name: count, dtype: int64

DBSCAN doesn't use 1/-1 the way Isolation Forest did. Instead:

0, 1, 2, etc. — each distinct number represents a different cluster DBSCAN found
-1 — this specifically means "noise" basicaly a point which didnt fit well in teh cluster.

In [31]:
## Re-running DBSCAN with a larger eps

dbscan_v2 = DBSCAN(eps=8, min_samples=10)
clusters_v2 = dbscan_v2.fit_predict(X_dbscan_scaled)

pd.Series(clusters_v2).value_counts()

 0     18951
 1       371
-1       279
 4       147
 2       131
 10       29
 5        22
 8        19
 3        16
 6        13
 7        12
 9        10
Name: count, dtype: int64

In [32]:
## Checking what's really inside cluster 0

check_df = pd.DataFrame({'cluster': clusters_v2, 'actual_label': y_dbscan.values})
pd.crosstab(check_df['cluster'], check_df['actual_label'])

actual_label,BENIGN,PortScan
cluster,,
-1,274,5
0,8984,9967
1,371,0
2,131,0
3,16,0
4,147,0
5,22,0
6,13,0
7,12,0


tried DBSCAN as an alternative to Isolation Forest, since my EDA suggested PortScan formed a dense cluster. But when I ran DBSCAN across all 78 features, it mostly merged BENIGN and PortScan into one shared cluster, rather than separating them clearly — even after adjusting eps. I think this suggests that while 2-3 individual features show a clear visual separation, the full feature set may need dimensionality reduction, or a more targeted subset of features, before clustering can separate them well. That feels like a good next direction to explor